# MASLD-LLM · 09 — MB-86xx freeze-16 re-evaluation

One-cell driver for `run_mb86xx_reeval.py` (Drive: masld-llm/run_mb86xx_reeval.py).

**Why this exists**: the freeze-8 MB-86xx corpus paired every item with the
previous patient's expression (header off-by-one). The four model archives
built on it are quarantined. This re-runs those cells on the corrected
freeze-16 corpus (21 patients, 57 records).

The script HARD-GATES on the Drive corpus being exactly freeze-16 (manifest
version + SHA + record count) before loading any model — if Drive holds a
stale copy it aborts instead of wasting GPU time.

Runtime: any GPU works (T4 fine) — 4 configs × 57 items ≈ 10–20 min total.
Run this BEFORE the 9B notebook (08) — it is quick and unblocks the
leaderboard's MB-86xx cells.

In [ ]:
# Cell 1: mount + run (resume-safe; re-running overwrites the 4 archives)
from google.colab import drive
drive.mount('/content/drive')
!python /content/drive/MyDrive/masld-llm/run_mb86xx_reeval.py

In [ ]:
# Cell 2: verification — all four archives present, on 21 patients
import gzip, json, glob
paths = sorted(glob.glob('/content/drive/MyDrive/masld-llm/results/frontier_runs/*__MB-86xx.jsonl.gz'))
assert len(paths) == 4, f'expected 4 archives, found {len(paths)}'
for p in paths:
    recs = [json.loads(l) for l in gzip.open(p, 'rt')]
    meta = recs[0]['__meta__']
    body = [r for r in recs if 'task' in r]
    print(p.split('/')[-1], '|', len(body), 'records |',
          len({r['sample'] for r in body}), 'patients | builder:', meta['builder'][:40])